# Exploring Data

In [ ]:
import sys
sys.path.append('/Users/luvsuneja/Documents/repos/advanced-rag-experimentation/')  # Go up one level
from setup import *

In [ ]:
RESTAURANT_REVIEWS_CSV = os.getenv("RESTAURANT_REVIEWS_CSV")

In [ ]:
reviews = pd.read_csv(RESTAURANT_REVIEWS_CSV)

In [ ]:
reviews.shape

In [ ]:
reviews['review'].str.len()

In [ ]:
# Display the full 'review' column without truncation
pd.set_option('display.max_colwidth', None)
reviews

In [ ]:
dimensions= ("price", "")

In [ ]:
print("🎯 IDENTIFIED 6 KEY QUERY DIMENSIONS:")
print()

dimensions = {
      "Price": ["Budget-friendly", "Mid-range",
   "Expensive ($80+)"],
      "👥 GROUP SIZE": ["Solo/Couple", "Small group (3-4)", "Large 
  group (5+)"],
      "🔊 NOISE LEVEL": ["Quiet/Work-friendly", "Moderate conversation",
  "Loud/Social"],
      "🍽️ CUISINE TYPE": ["Italian", "Asian", "American/Cafe"],
      "⭐ OCCASION": ["Casual dining", "Special occasion",
  "Work/Business"],
      "🕒 TIME SENSITIVITY": ["Quick bite", "Leisurely meal", "Extended 
  stay"]
}

In [ ]:
simple_questions = {
    "simple_queries": [
      {
        "query_id": 1,
        "query": "Italian restaurant",
        "expected_results": [
          {
            "restaurant_id": 1,
            "restaurant": "Mario's Bistro",
            "relevance": "highly_relevant",
            "reason": "Explicitly Italian cuisine, authentic pasta"
          },
          {
            "restaurant_id": 5,
            "restaurant": "Tony's Italian Kitchen",
            "relevance": "highly_relevant",
            "reason": "Italian restaurant with authentic food and wine list"
          }
        ]
      },
      {
        "query_id": 2,
        "query": "sushi place",
        "expected_results": [
          {
            "restaurant_id": 2,
            "restaurant": "Sakura Sushi",
            "relevance": "highly_relevant",
            "reason": "Sushi restaurant with vegetarian options"
          },
          {
            "restaurant_id": 8,
            "restaurant": "Neko Sushi",
            "relevance": "highly_relevant",
            "reason": "Exclusive sushi experience, omakase style"
          }
        ]
      },
      {
        "query_id": 3,
        "query": "coffee shop",
        "expected_results": [
          {
            "restaurant_id": 3,
            "restaurant": "Code & Coffee",
            "relevance": "highly_relevant",
            "reason": "Coffee shop with work-friendly environment"
          },
          {
            "restaurant_id": 6,
            "restaurant": "Starbucks Downtown",
            "relevance": "relevant",
            "reason": "Coffee chain, though not work-friendly"
          }
        ]
      },
      {
        "query_id": 4,
        "query": "expensive dining",
        "expected_results": [
          {
            "restaurant_id": 4,
            "restaurant": "Le Bernardin SF",
            "relevance": "highly_relevant",
            "reason": "$220 per person tasting menu"
          },
          {
            "restaurant_id": 8,
            "restaurant": "Neko Sushi",
            "relevance": "highly_relevant",
            "reason": "$150 per person minimum, exclusive experience"
          }
        ]
      },
      {
        "query_id": 5,
        "query": "family restaurant",
        "query_type": "simple_keyword",
        "expected_results": [
          {
            "restaurant_id": 7,
            "restaurant": "Giuseppe's Pizza",
            "relevance": "highly_relevant",
            "reason": "Perfect for big groups, family of 8, kids running around"
          },
          {
            "restaurant_id": 12,
            "restaurant": "Family Garden",
            "relevance": "highly_relevant",
            "reason": "Family-friendly, huge booths, kids menu, casual atmosphere"
          }
        ]
      }
    ],
    "metadata": {
      "description": "Simple keyword queries that both Dense and ColBERT retrieval can handle well",
      "expected_performance": "tie",
      "note": "These queries have clear keyword matches without complex logic"
    }
  }

In [ ]:
import json
import os


# Ensure the output directory exists
output_dir = os.getenv("DATA_DIR")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "simple_questions.json")

# Save the data as JSON
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(simple_questions, f, indent=2, ensure_ascii=False)

print(f"Saved simple questions to {output_path}")


In [ ]:
complex_questions = {
    "complex_queries": [
        {
            "query_id": 6,
            "query": "quiet wifi laptop work under $30",
            "query_type": "multi_constraint",
            "expected_colbert_winner": {
                "restaurant_id": 3,
                "restaurant": "Code & Coffee",
                "relevance": "highly_relevant",
                "reason": "Matches ALL constraints: quiet zones✓ blazing wifi✓ work-friendly✓ under $30✓",
                "supporting_text": "dedicated quiet zones, blazing fast wifi (100+ Mbps), comfortable chairs, been coming here daily for 3 months"
            },
            "why_dense_struggles": "Cannot match multiple simultaneous constraints - might return expensive work places or cheap non-work places",
            "dense_likely_confusion": [
                "Starbucks Downtown (wifi but NOT quiet)",
                "Green Tea House (quiet but NO wifi)"
            ]
        },
        {
            "query_id": 7,
            "query": "NOT noisy NOT expensive good for work",
            "query_type": "negation_logic",
            "expected_colbert_winner": {
                "restaurant_id": 3,
                "restaurant": "Code & Coffee",
                "relevance": "highly_relevant",
                "reason": "NOT noisy (quiet zones)✓ NOT expensive (reasonable)✓ work-friendly✓",
                "supporting_text": "dedicated quiet zones, completely silent floor upstairs, exceptional coffee, daily for 3 months"
            },
            "why_dense_struggles": "Dense embeddings cannot handle negation logic - 'NOT noisy' confuses single-vector similarity",
            "dense_likely_confusion": [
                "Might return expensive quiet places",
                "Cannot process NOT logic effectively"
            ]
        },
        {
            "query_id": 8,
            "query": "expensive but worth every penny special occasion",
            "query_type": "contradictory_context",
            "expected_colbert_winner": {
                "restaurant_id": 4,
                "restaurant": "Le Bernardin SF",
                "relevance": "highly_relevant",
                "reason": "Expensive BUT explicitly worth it for special occasions",
                "supporting_text": "EXPENSIVE $220 per person BUT worth every single penny, once-in-a-lifetime dining experience"
            },
            "why_dense_struggles": "Conflicting signals: 'expensive' vs 'worth it' - dense vector averages out the contradiction",
            "dense_likely_confusion": [
                "Might avoid due to 'expensive' signal",
                "Cannot understand expensive BUT worth it paradox"
            ]
        },
        {
            "query_id": 9,
            "query": "large family groups kids running around loud atmosphere",
            "query_type": "multi_constraint",
            "expected_colbert_winner": {
                "restaurant_id": 7,
                "restaurant": "Giuseppe's Pizza",
                "relevance": "highly_relevant",
                "reason": "Perfect for large groups✓ kids welcome✓ loud/lively  atmosphere✓",
                "supporting_text": "8 people total, kids were running around andnobody cared, lively and loud in a good way, big groups"
            },
            "why_dense_struggles": "Multiple complex requirements - dense retrieval loses nuance in single similarity score",
            "dense_likely_confusion": [
                "Might return quiet family places",
                "Cannot handle 'loud atmosphere' as positive"
            ]
        },
        {
            "query_id": 10,
            "query": "vegetarian options but NOT expensive",
            "query_type": "constraint_with_negation",
            "expected_colbert_winner": {
                "restaurant_id": 2,
                "restaurant": "Sakura Sushi",
                "relevance": "highly_relevant",
                "reason": "Great vegetarian options✓ NOT expensive ($25 per person)✓",
                "supporting_text": "caters to vegetarians, amazing tempura vegetable roll, reasonable prices around $25 per person"
            },
            "why_dense_struggles": "Cannot effectively combine positive constraint (vegetarian) with negation (NOT expensive)",
            "dense_likely_confusion": [
                "Might return expensive vegetarian places",
                "Negation logic fails in embeddings"
            ]
        },
        {
            "query_id": 11,
            "query": "work laptop but NOT like Starbucks chaos",
            "query_type": "comparative_negation",
            "expected_colbert_winner": {
                "restaurant_id": 3,
                "restaurant": "Code & Coffee",
                "relevance": "highly_relevant",
                "reason": "Work-friendly✓ laptop-suitable✓ NOT chaotic like Starbucks✓",
                "supporting_text": "perfect work spots, quiet zones, completely silent floor - opposite of Starbucks chaos"
            },
            "why_dense_struggles": "Cannot process comparative negation - 'NOT like Starbucks' requires understanding specific comparison",
            "dense_likely_confusion": [
                "Might still return Starbucks",
                "Cannot process comparative NOT logic"
            ],
            "anti_example": {
                "restaurant_id": 6,
                "restaurant": "Starbucks Downtown",
                "why_wrong": "Explicitly described as 'absolute chaos' - exactly what query wants to avoid"
            }
        }
    ],
    "metadata": {
        "description": "Complex queries where ColBERT's token-level matching should dominate Dense retrieval",
        "expected_performance": "colbert_wins",
        "key_advantages": [
            "Multi-constraint handling",
            "Negation logic understanding",
            "Contradictory requirement resolution",
            "Comparative reasoning"
        ],
        "dense_weakness": "Single similarity vector cannot handle complex logical combinations"
    }
}

import os
import json

DATA_DIR = os.getenv('DATA_DIR')
with open(os.path.join(DATA_DIR, "complex_questions.json"), "w", encoding="utf-8") as f:
    json.dump(complex_questions, f, indent=2, ensure_ascii=False)


In [ ]:
os.path.join(DATA_DIR, "complex_questions.json")